<a href="https://colab.research.google.com/github/anjitha2002/Rag_Chatbot_Using_Langchain/blob/main/Kerala_Development_Analysis_Rag_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q langchain-community langchain-huggingface langchain-chroma pdfplumber langchain-google-genai sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil

In [4]:
import os
from google.colab import userdata,files

os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLEAI_API_KEY")


# Document Loading and Extracting

In [5]:
uploaded=files.upload()


Saving kerala-development-report_2026.pdf to kerala-development-report_2026.pdf


In [6]:
import pdfplumber
from langchain_core.documents import Document


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:

pdf_files = list(uploaded.keys())

In [9]:
docs=[]
for pdf_path in pdf_files:
  with pdfplumber.open(pdf_path) as pdf:
    for i,page in enumerate(pdf.pages):
      text=page.extract_text(layout=True) or""
      docs.append(Document(page_content=text,metadata={"source":pdf_path,"page":i+1}))
print("Total pages:",len(docs))

Total pages: 408


# Splitting

In [13]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=350,
)
chunks=splitter.split_documents(docs)

In [14]:
print("total chunks:",len(chunks))

total chunks: 1200


In [15]:
print(chunks[150].metadata)

{'source': 'kerala-development-report_2026.pdf', 'page': 57}


In [16]:
print(chunks[60].page_content)

INTRODUCTION | 13    
                                                                                  
        artists across 45 art forms, women-led initiatives like youth. Together, these interventions demonstrate
        Samam promoting gender equality in culture, and large- how Kerala has leveraged culture, sports, and youth
        scale relief during Covid-19 reaching over 60,000 artists. development as modes of inclusion, employment,
        Kerala also undertook a transformative push in sports global recognition, and long-term social progress.
        and youth affairs. Plan outlays for sports rose by nearly                 
        160 per cent between 2016–17 and 2026–27, backed The Government of Kerala is thus committed to
        by over Rs 3,500 crore for infrastructure, including widening and deepening its investments in the people
        new District stadiums, synthetic tracks, academies, and and to economic growth. It will do so while maintaining
        India’s firs

# Vector Database Setup

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [18]:
emb=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
db=Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    persist_directory="/content/chroma_db"
)

# Retriever

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [21]:
retriever=db.as_retriever(search_kwargs={"k":10})


# Prompt Setup

In [22]:
prompt=ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question using only the context below.
If the answer is not in the context,say "i dont know"

Context:
{context}

Question:
{question}
"""
)

# LLM and Rag Chain Setup

In [23]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",temperature=0)

In [24]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain=(
    {
        "context":retriever|format_docs,
        "question":RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
  )

# Chat Interface

In [25]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

chat_history = []

while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nBot:", answer, "\n")

    chat_history.append({
        "question": question,
        "answer": answer
    })

You: what all topics are covered in this report

Bot: Based on the provided context, the report covers the following topics (chapters/sections and initiatives):

* Local Governments (including Decentralisation and Local Self-Governments in Kerala, and Local Self-Government Programmes for Livelihood Security and Poverty Alleviation)
* Urban Development
* Environment and Growth (including Global Warming and Climate Action: A Perspective for Kerala, Learning from Disaster: The Kerala Experience, and Waste Management)
* Transport for Modern Economy and Sustainable Growth
* Sustainable Energy Infrastructure
* State, Migrants, and a Shared Future: A Decade of Migrant-Centric Interventions in Kerala
* Human Resources: Unleashing the Potential
* Arts and Culture
* Sports and Youth Affairs
* Digital Surveys and Land Records
* Initiatives for a New Kerala (including Vidyakiranam Mission, Haritha Keralam Mission, LIFE Mission, and Rebuild Kerala Initiative)
* Special Development Packages in Keral

# Pdf Generation

In [26]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.9 MB/s  0:00:00


In [27]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from xml.sax.saxutils import escape
from google.colab import files

def save_chat_pdf():

    doc = SimpleDocTemplate("chat_history.pdf")
    styles = getSampleStyleSheet()

    content = []

    content.append(Paragraph("Kerala Development Report Rag ChatBot Chat History", styles["Title"]))
    content.append(Spacer(1, 20))

    for chat in chat_history:

        content.append(Paragraph(
            "<b>Question:</b> " + escape(chat["question"]),
            styles["Normal"]
        ))

        content.append(Spacer(1, 10))

        content.append(Paragraph(
            "<b>Answer:</b> " + escape(chat["answer"]).replace("\n", "<br/>"),
            styles["Normal"]
        ))

        content.append(Spacer(1, 20))

    doc.build(content)

    files.download("chat_history.pdf")


save_chat_pdf()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>